## Representation learning baselines

Frozen **wav2vec 2.0**, **HuBERT**, and **Whisper** (encoder only) embeddings with a linear probe (**StandardScaler + LogisticRegression**) and **GroupKFold** grouped like the other model notebooks (`file_stem` for Androids, `participant_id` for RADAR). Metrics are saved under `results/metrics/repr_learn/`.

**Dependencies**: install once in a terminal (`pip install -r requirements.txt` or `pip install "transformers>=4.40,<5"`) so you get **transformers 4.x** (v5 is untested here). **Do not run `pip install` inside this notebook:** upgrading packages in a live Jupyter kernel often **crashes the kernel on Windows** because NumPy/Torch DLLs get out of sync. After any env change, use **Restart Kernel** before running model code.

If the kernel still dies: set `FORCE_CPU = True` in the code cell (GPU VRAM), set `BATCH_SIZE = 1`, and close other GPU apps.

**RADAR audio**: the processed RADAR CSV only stores `File` (e.g. `20201230_1100-scripted-1-1.wav`). Set `RADAR_AUDIO_ROOT` below to a folder whose tree contains those `.wav` files (files are resolved by name). If wavs are not available, the RADAR block will skip missing files automatically.

**Cell 2 (below)** trains a **small PyTorch MLP head** on pooled pretrained **Wav2Vec2 or HuBERT** frame outputs (encoder **weights frozen**; only the head updates). Requires **cell 1** to have been run first (shared `DEVICE`, `load_waveform_mono`, `MODEL_IDS`, etc.). Results: `repr_learn/androids_*_dl_head_cv_folds.csv`.

______________________________________________________________________________________________________

To change: 

- RADAR_AUDIO_ROOT (RADAR .wav files)
- RADAR_CSV (RADAR metadata/label CSV that contains the audio filename and target label information)
- prepare_radar() (Must match  actual RADAR CSV column names and label structure)
- "File" column (audio filename)
- "participant_id" column (RADAR participant identifier column)
- Must match your RADAR depression score column, or be replaced with another target column.
- df["depressed"] = (df["phq8_score"] >= 10).astype(int) (Must be changed if you are not doing binary PHQ-8 ≥ 10 classification)
- resolve() inside prepare_radar() (Must be changed if the filenames in the CSV do not exactly match the .wav filenames)
- MAX_AUDIO_SECONDS (Must be reconsidered, because the current code only uses the first 30 seconds of each RADAR recording)

In [ ]:
from __future__ import annotations

import gc
import os
from pathlib import Path
from typing import Callable

# Avoid oversubscribing CPU threads (occasionally unstable in Jupyter on Windows)
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

import librosa
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from transformers import (
    HubertModel,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Model,
    WhisperModel,
    WhisperProcessor,
)

# ---------------------------------------------------------------------------
# Paths (match other notebooks)
# ---------------------------------------------------------------------------

# radar_audio: may need changing if running on a different computer/path.
PROJECT = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project")

# radar_audio: Androids-specific processed CSV.
# Leave out / replace if this notebook becomes RADAR-only.
ANDROID_CSV = PROJECT / "data/processed/androids_model_dataset_basic.csv"

# radar_audio: main RADAR metadata/features CSV.
# For raw RADAR audio, this must contain at least File, participant_id, and phq8_score/depression label.
RADAR_CSV = PROJECT / "data/processed/radar_model_dataset_raw_features.csv"

# radar_audio: results folder may need renaming if outputs are specifically for raw RADAR audio.
# Example: PROJECT / "results/metrics/repr_learn_radar_audio"
RESULTS_PATH = PROJECT / "results/metrics/repr_learn"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Point this at the directory tree that holds RADAR WAVs (recursive lookup by File name).
# radar_audio: this is one of the most important paths to update.
# Example guesses — adjust until `Path.exists()` resolves your data:
RADAR_AUDIO_ROOT = PROJECT / "datasets/RADAR-MDD"

# --- stability (kernel dies on GPU OOM / Windows DLL mismatch after pip in-notebook) ---

# radar_audio: set to True if RADAR raw audio causes CUDA/GPU crashes.
FORCE_CPU = False

# radar_audio: raw audio embedding extraction is memory-heavy.
# Keep low for long RADAR clips; increase only if GPU memory allows.
BATCH_SIZE = 1

torch.set_num_threads(min(8, max(1, os.cpu_count() or 1)))
if not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
else:
    DEVICE = torch.device("cpu")

# radar_audio: keep 16000 for wav2vec2/HuBERT/Whisper unless model requirements change.
TARGET_SR = 16000

# radar_audio: likely needs changing/chunking for RADAR if recordings are longer than 30 seconds.
MAX_AUDIO_SECONDS = 30

# radar_audio: may need lowering if RADAR has too few participants/classes per fold.
N_FOLDS = 5

# radar_audio: can stay the same, but you may want to test only one model first for speed.
MODEL_IDS = {
    "wav2vec2": "facebook/wav2vec2-base",
    "hubert": "facebook/hubert-base-ls960",
    "whisper": "openai/whisper-base",
}


def _mean_pool(hidden: torch.Tensor, mask: torch.Tensor | None) -> torch.Tensor:
    # radar_audio: probably unchanged.
    # This pools frame-level transformer outputs into one vector per recording.
    if mask is None:
        return hidden.mean(dim=1)
    m = mask.unsqueeze(-1).to(hidden.dtype)
    summed = (hidden * m).sum(dim=1)
    denom = m.sum(dim=1).clamp(min=1e-6)
    return summed / denom


def build_wav_filename_index(audio_root: Path) -> dict[str, Path]:
    # radar_audio: likely important.
    # This assumes RADAR metadata filenames match actual .wav filenames exactly.
    # If filenames differ, this function needs stronger matching logic.
    if not audio_root.exists():
        return {}

    idx: dict[str, Path] = {}

    # radar_audio: recursive search may be slow if RADAR_AUDIO_ROOT is too broad.
    for p in audio_root.rglob("*.wav"):
        idx.setdefault(p.name, p)
        idx.setdefault(p.name.lower(), p)

    return idx


def load_waveform_mono(path: Path, target_sr: int) -> np.ndarray:
    # radar_audio: likely to change if you want chunking instead of simple truncation.
    wav, sr = librosa.load(str(path), sr=target_sr, mono=True)

    # radar_audio: currently keeps only the first MAX_AUDIO_SECONDS.
    # For RADAR, consider random crops, multiple chunks per file, or full-file aggregation.
    max_len = int(MAX_AUDIO_SECONDS * target_sr)
    if len(wav) > max_len:
        wav = wav[:max_len]

    return wav.astype(np.float32)


def encode_wav2vec_family(paths: list[Path], model_id: str, batch_size: int | None = None) -> np.ndarray:
    # radar_audio: mostly reusable.
    # The main things likely to change are batch_size and load_waveform_mono().
    bs = BATCH_SIZE if batch_size is None else batch_size

    processor = Wav2Vec2FeatureExtractor.from_pretrained(model_id)

    if "hubert" in model_id.lower():
        model = HubertModel.from_pretrained(model_id, low_cpu_mem_usage=True)
    else:
        model = Wav2Vec2Model.from_pretrained(model_id, low_cpu_mem_usage=True)

    model.to(DEVICE)
    model.eval()

    out_list: list[np.ndarray] = []

    for start in range(0, len(paths), bs):
        batch_paths = paths[start : start + bs]

        # radar_audio: depends on load_waveform_mono().
        # If RADAR files are long, this currently embeds only the first 30 seconds.
        waves = [load_waveform_mono(p, TARGET_SR) for p in batch_paths]

        feats = processor(
            waves,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
        )

        feats = {k: v.to(DEVICE) for k, v in feats.items()}

        with torch.inference_mode():
            outputs = model(**feats)

        pooled = _mean_pool(outputs.last_hidden_state, feats.get("attention_mask"))
        out_list.append(pooled.cpu().numpy())

    model.cpu()
    del model, processor
    gc.collect()

    # radar_audio: may error if CUDA unavailable; safer version would check torch.cuda.is_available().
    torch.cuda.empty_cache()

    return np.vstack(out_list)

def encode_whisper(paths: list[Path], model_id: str, batch_size: int | None = None) -> np.ndarray:
    # radar_audio: mostly reusable.
    # Main RADAR-specific issue is still load_waveform_mono(), because long recordings
    # are currently truncated to MAX_AUDIO_SECONDS.
    bs = BATCH_SIZE if batch_size is None else batch_size

    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperModel.from_pretrained(model_id, low_cpu_mem_usage=True)
    model.to(DEVICE)
    model.eval()

    out_list: list[np.ndarray] = []

    for start in range(0, len(paths), bs):
        batch_paths = paths[start : start + bs]

        # radar_audio: currently loads one fixed-length waveform per file.
        # For RADAR, consider chunking each file and averaging embeddings across chunks.
        waves = [load_waveform_mono(p, TARGET_SR) for p in batch_paths]

        inputs = processor(
            waves,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
        )

        input_features = inputs.input_features.to(DEVICE)

        with torch.inference_mode():
            enc = model.encoder(input_features)
            hidden = enc.last_hidden_state

        # radar_audio: simple mean pooling across Whisper encoder frames.
        # Can stay unchanged unless you want attention/statistical pooling.
        pooled = hidden.mean(dim=1)

        out_list.append(pooled.cpu().numpy())

    model.cpu()
    del model, processor
    gc.collect()

    # radar_audio: safer to guard this if CUDA is unavailable.
    # Example: if torch.cuda.is_available(): torch.cuda.empty_cache()
    torch.cuda.empty_cache()

    return np.vstack(out_list)


ENCODERS: dict[str, Callable[[list[Path], str, int], np.ndarray]] = {
    # radar_audio: reusable. You can temporarily comment out slower models
    # when testing the RADAR pipeline.
    "wav2vec2": encode_wav2vec_family,
    "hubert": encode_wav2vec_family,
    "whisper": encode_whisper,
}


def align_embeddings(paths: pd.Series, unique_paths: np.ndarray, mat: np.ndarray) -> np.ndarray:
    # radar_audio: reusable.
    # This maps unique audio-level embeddings back onto the row-level dataframe.
    # Important if RADAR has repeated rows pointing to the same audio file.
    lookup = {str(p): i for i, p in enumerate(unique_paths)}
    idx = np.array([lookup[str(p)] for p in paths], dtype=np.int64)
    return mat[idx]


def run_group_cv(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    n_folds: int = N_FOLDS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # radar_audio: currently assumes binary classification with grouped CV.
    # If using PHQ-8 regression, replace LogisticRegression and classification metrics.
    # If using official RADAR train/test split, this function may be replaced or supplemented.

    gkf = GroupKFold(n_splits=n_folds)
    fold_rows: list[dict] = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pipe = Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=5000,
                        class_weight="balanced",  # radar_audio: keep for imbalanced binary labels
                        solver="lbfgs",
                        random_state=42,
                    ),
                ),
            ]
        )

        pipe.fit(X[tr], y[tr])

        pred = pipe.predict(X[te])
        proba = pipe.predict_proba(X[te])[:, 1]

        # radar_audio: dummy baseline is reusable for binary classification.
        # For regression, replace with DummyRegressor.
        dummy = DummyClassifier(strategy="stratified", random_state=42)
        dummy.fit(X[tr], y[tr])
        p_dummy = dummy.predict_proba(X[te])[:, 1]

        # radar_audio: Wilcoxon currently compares absolute probability errors.
        # For regression, compare absolute PHQ-8 prediction errors instead.
        e_m = np.abs(y[te] - proba)
        e_d = np.abs(y[te] - p_dummy)

        if len(e_m) >= 2 and not np.allclose(e_m, e_d):
            w_p = float(wilcoxon(e_m, e_d, zero_method="wilcox", mode="auto").pvalue)
        else:
            w_p = float("nan")

        fold_rows.append(
            {
                "fold": fold,
                "n_train": len(tr),
                "n_test": len(te),

                # radar_audio: binary classification metrics.
                # Replace with MAE/RMSE/R2 if using continuous PHQ-8.
                "accuracy": accuracy_score(y[te], pred),
                "f1": f1_score(y[te], pred, zero_division=0),
                "roc_auc": roc_auc_score(y[te], proba),

                "wilcoxon_p_vs_stratified_dummy": w_p,
            }
        )

    folds_df = pd.DataFrame(fold_rows)

    summary = pd.DataFrame(
        [
            {
                "subset": "all",
                "n_rows": len(y),
                "n_groups": len(np.unique(groups)),

                # radar_audio: classification summary metrics.
                # Replace if modelling PHQ-8 as regression.
                "accuracy_mean": folds_df["accuracy"].mean(),
                "accuracy_std": folds_df["accuracy"].std(),
                "f1_mean": folds_df["f1"].mean(),
                "f1_std": folds_df["f1"].std(),
                "roc_auc_mean": folds_df["roc_auc"].mean(),
                "roc_auc_std": folds_df["roc_auc"].std(),
            }
        ]
    )

    return folds_df, summary


def run_repr_for_dataset(
    dataset_key: str,
    manifest: pd.DataFrame,
    path_series: pd.Series,
    groups: pd.Series,
    y: pd.Series,
) -> None:
    # radar_audio: reusable wrapper, but assumes each row has:
    # audio path, group ID, and binary target y.

    valid = (
        path_series.notna()
        & groups.notna()
        & y.notna()
        & path_series.astype(str).str.len().gt(0)
    )

    df = manifest.loc[valid].copy()
    paths = path_series.loc[valid].astype(str)

    # radar_audio: groups should usually be participant_id to prevent leakage.
    grp = groups.loc[valid].astype(str).values

    # radar_audio: currently casts target to int, suitable for binary depression.
    # For PHQ-8 regression, remove .astype(int).
    yt = y.loc[valid].astype(int).values

    uniq = pd.unique(paths)
    uniq_paths = np.array([Path(p) for p in uniq])

    print(f"{dataset_key}: {len(paths)} usable rows | {len(uniq_paths)} unique audio files")

    for model_name, encoder in ENCODERS.items():
        print(f"  -> embeddings: {model_name} ({MODEL_IDS[model_name]})")

        # radar_audio: this may take a long time for raw RADAR audio.
        emb_uniq = encoder(uniq_paths.tolist(), MODEL_IDS[model_name])

        X = align_embeddings(paths, uniq, emb_uniq)

        folds_df, summary_df = run_group_cv(X, yt, grp)

        # radar_audio: output names depend on dataset_key.
        # Use dataset_key="radar_audio" if you want clearer filenames.
        folds_path = RESULTS_PATH / f"{dataset_key}_{model_name}_cv_folds.csv"
        summary_path = RESULTS_PATH / f"{dataset_key}_{model_name}_summary.csv"

        folds_df.to_csv(folds_path, index=False)
        summary_df.to_csv(summary_path, index=False)

        print(f"     saved: {folds_path.name}, {summary_path.name}")


def prepare_androids() -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    # radar_audio: Androids-specific preparation.
    # Remove/comment this function if the notebook becomes RADAR-only.

    df = pd.read_csv(ANDROID_CSV)

    # radar_audio: Androids-specific columns.
    df["file_path"] = df["file_path"].astype(str).str.strip()
    df["file_stem"] = df["file_stem"].astype(str).str.strip()
    df["depressed"] = pd.to_numeric(df["depressed"], errors="coerce")

    path_series = df["file_path"].map(lambda p: Path(p))
    exists = path_series.map(lambda p: p.is_file())

    df = df.loc[exists].copy()
    path_series = path_series.loc[exists]

    # radar_audio: Androids uses file_stem as grouping variable here.
    # For RADAR, use participant_id instead.
    return df, path_series, df["file_stem"], df["depressed"]


def prepare_radar(wav_index: dict[str, Path]) -> tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series] | None:
    # radar_audio: this is the main function to adapt for raw RADAR audio.

    df = pd.read_csv(RADAR_CSV)

    # radar_audio: these columns must exist in the RADAR CSV.
    # Rename here if your file uses different names.
    df["participant_id"] = df["participant_id"].astype(str).str.strip()
    df["phq8_score"] = pd.to_numeric(df["phq8_score"], errors="coerce")

    # radar_audio: currently drops rows missing PHQ-8, File, or participant_id.
    df = df.dropna(subset=["phq8_score", "File", "participant_id"]).copy()

    # radar_audio: binary depression threshold.
    # Keep if classification; remove/change if predicting continuous PHQ-8.
    df["depressed"] = (df["phq8_score"] >= 10).astype(int)

    # radar_audio: assumes RADAR CSV column is called "File".
    names = df["File"].astype(str).str.strip()

    def resolve(name: str) -> Path | None:
        # radar_audio: filename matching logic.
        # May need changing if metadata names do not exactly match .wav filenames.
        if not name:
            return None

        if name in wav_index:
            return wav_index[name]

        low = name.lower()
        return wav_index.get(low)

    audio_paths = names.map(resolve)
    n_ok = audio_paths.notna().sum()

    if n_ok == 0:
        # radar_audio: this warning usually means RADAR_AUDIO_ROOT is wrong
        # or filenames in the CSV do not match actual .wav names.
        print(
            "RADAR: could not resolve any WAV paths. Set RADAR_AUDIO_ROOT so it contains",
            "the recording files referenced in column 'File'.",
        )
        return None

    df = df.loc[audio_paths.notna()].copy()
    paths_series = audio_paths.loc[audio_paths.notna()].map(lambda p: Path(p))

    if n_ok < len(names):
        # radar_audio: useful diagnostic for missing WAVs.
        print(f"RADAR: using {n_ok} / {len(names)} rows with found audio files")

    # radar_audio: returns manifest, audio paths, grouping variable, and binary target.
    # For regression, return df["phq8_score"] instead of df["depressed"].
    return df, paths_series, df["participant_id"], df["depressed"]


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------



# radar_audio: builds lookup of actual RADAR .wav files.
# This depends heavily on RADAR_AUDIO_ROOT being correct.
RADAR_IDX = build_wav_filename_index(RADAR_AUDIO_ROOT)

# radar_audio: prepares RADAR dataframe by matching metadata File names to WAV paths.
rad = prepare_radar(RADAR_IDX)

if rad is not None:
    radar_df, r_paths, r_groups, r_y = rad

    # radar_audio: consider changing dataset_key to "radar_audio"
    # so the output filenames are clearer.
    run_repr_for_dataset("radar", radar_df, r_paths, r_groups, r_y)

print("Done. Outputs in:", RESULTS_PATH)

In [ ]:
# ---------------------------------------------------------------------------
# Simple DL: frozen Wav2Vec2 / HuBERT trunk + trainable 2-layer MLP head
# No folds: single group-aware train/test split
#
# Run your first setup cell first so these already exist:
# DEVICE, MODEL_IDS, prepare_androids, load_waveform_mono,
# RESULTS_PATH, TARGET_SR
# ---------------------------------------------------------------------------

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from transformers import HubertModel, Wav2Vec2FeatureExtractor, Wav2Vec2Model


try:
    DEVICE, MODEL_IDS, prepare_androids, load_waveform_mono, RESULTS_PATH, TARGET_SR
except NameError as e:
    raise RuntimeError(
        "Run the previous notebook cell first: imports, paths, DEVICE, MODEL_IDS, "
        "prepare_androids, load_waveform_mono, RESULTS_PATH, TARGET_SR."
    ) from e


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

DL_BACKBONE = "wav2vec2"  # radar_audio: can remain unchanged unless comparing different backbones
DL_EPOCHS = 8             # radar_audio: may require tuning
DL_BATCH = 4              # radar_audio: may need reducing for longer recordings
DL_LR = 1e-3              # radar_audio: may require tuning
DL_NUM_WORKERS = 0
TEST_SIZE = 0.2           # radar_audio: remove if using official RADAR split
RANDOM_STATE = 42


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

class FrozenEncoderClassifier(nn.Module):
    """
    Frozen Wav2Vec2 / HuBERT encoder + trainable MLP classification head.
    """

    def __init__(self, backbone_key: str) -> None:
        super().__init__()

        model_id = MODEL_IDS[backbone_key]

        if backbone_key == "hubert":
            self.encoder = HubertModel.from_pretrained(
                model_id,
                low_cpu_mem_usage=True,
            )
        else:
            self.encoder = Wav2Vec2Model.from_pretrained(
                model_id,
                low_cpu_mem_usage=True,
            )

        # Freeze representation model
        for p in self.encoder.parameters():
            p.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(
        self,
        input_values: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        out = self.encoder(
            input_values=input_values,
            attention_mask=attention_mask,
        ).last_hidden_state

        # Do NOT use the raw audio attention mask for pooling here.
        # The encoder output is downsampled, so hidden length != raw audio length.
        pooled = out.mean(dim=1)

        logits = self.head(pooled).squeeze(-1)
        return logits


# ---------------------------------------------------------------------------
# Dataset + collate
# ---------------------------------------------------------------------------

class AudioDataset(Dataset):
    def __init__(self, path_strings: list[str], labels: np.ndarray) -> None:
        self.paths = path_strings

        # radar_audio: labels currently assumed to be binary depression labels.
        # Replace if using PHQ-8 regression.
        self.labels = labels.astype(np.float32)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> tuple[np.ndarray, float]:
        wav = load_waveform_mono(Path(self.paths[idx]), TARGET_SR)
        label = float(self.labels[idx])
        return wav, label


def dl_collate(
    batch: list[tuple[np.ndarray, float]],
    processor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    waves, labels = zip(*batch)
    wave_list = list(waves)

    try:
        feats = processor(
            wave_list,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
            return_attention_mask=True,
        )
    except TypeError:
        feats = processor(
            wave_list,
            sampling_rate=TARGET_SR,
            padding=True,
            return_tensors="pt",
        )

    input_values = feats["input_values"]

    if "attention_mask" in feats:
        attention_mask = feats["attention_mask"]
    else:
        lengths = [len(w) for w in wave_list]
        batch_size, max_len = input_values.shape

        attention_mask = torch.zeros(
            batch_size,
            max_len,
            dtype=torch.long,
        )

        for i, length in enumerate(lengths):
            attention_mask[i, : min(length, max_len)] = 1

    y = torch.tensor(labels, dtype=torch.float32)

    return input_values, attention_mask, y


# ---------------------------------------------------------------------------
# Train / evaluate helpers
# ---------------------------------------------------------------------------

def train_one_epoch(
    model: FrozenEncoderClassifier,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
) -> float:
    model.train()

    total_loss = 0.0
    n_samples = 0

    for input_values, attention_mask, yb in loader:
        input_values = input_values.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_values, attention_mask)
        loss = loss_fn(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += float(loss.detach().cpu()) * yb.size(0)
        n_samples += yb.size(0)

    return total_loss / max(n_samples, 1)


@torch.inference_mode()
def evaluate_model(
    model: FrozenEncoderClassifier,
    loader: DataLoader,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()

    all_logits = []
    all_labels = []

    for input_values, attention_mask, yb in loader:
        input_values = input_values.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_values, attention_mask)

        all_logits.append(logits.cpu().numpy())
        all_labels.append(yb.numpy())

    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)

    return logits, labels


# ---------------------------------------------------------------------------
# Main run function: no folds
# ---------------------------------------------------------------------------

def run_dl_androids_no_folds() -> None:
    # radar_audio: rename to run_dl_radar_no_folds()

    # radar_audio: replace prepare_androids() with prepare_radar().
    df, paths, groups, y = prepare_androids()

    valid = paths.notna() & groups.notna() & y.notna()

    path_arr = paths.loc[valid].astype(str).values

    # radar_audio: remove astype(int) if predicting continuous PHQ-8.
    y_arr = y.loc[valid].astype(int).values

    # radar_audio: groups should remain participant_id.
    group_arr = groups.loc[valid].astype(str).values

    print(f"Usable rows: {len(y_arr)}")
    print(f"Unique groups: {len(np.unique(group_arr))}")

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    # radar_audio: replace if using an official RADAR train/test split.

    train_idx, test_idx = next(
        splitter.split(
            np.zeros(len(y_arr)),
            y_arr,
            groups=group_arr,
        )
    )

    print(f"Train rows: {len(train_idx)}")
    print(f"Test rows: {len(test_idx)}")
    print(f"Train groups: {len(np.unique(group_arr[train_idx]))}")
    print(f"Test groups: {len(np.unique(group_arr[test_idx]))}")

    processor = Wav2Vec2FeatureExtractor.from_pretrained(
        MODEL_IDS[DL_BACKBONE]
    )

    train_ds = AudioDataset(
        path_arr[train_idx].tolist(),
        y_arr[train_idx],
    )

    test_ds = AudioDataset(
        path_arr[test_idx].tolist(),
        y_arr[test_idx],
    )

    collate_fn = lambda batch: dl_collate(batch, processor)

    train_loader = DataLoader(
        train_ds,
        batch_size=DL_BATCH,
        shuffle=True,
        num_workers=DL_NUM_WORKERS,
        collate_fn=collate_fn,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=DL_BATCH,
        shuffle=False,
        num_workers=DL_NUM_WORKERS,
        collate_fn=collate_fn,
    )

    model = FrozenEncoderClassifier(DL_BACKBONE).to(DEVICE)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=DL_LR,
    )

    y_train = y_arr[train_idx]

    # radar_audio: only needed for binary classification.
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    pos_weight = torch.tensor(
        [n_neg / max(n_pos, 1)],
        dtype=torch.float32,
        device=DEVICE,
    )

    # radar_audio: replace with nn.MSELoss() or nn.SmoothL1Loss()
    # if predicting PHQ-8 instead of binary depression.
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(1, DL_EPOCHS + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_fn,
        )

        print(
            f"Epoch {epoch}/{DL_EPOCHS} "
            f"train_loss={train_loss:.4f}"
        )

    logits_test, y_test = evaluate_model(model, test_loader)

    # radar_audio: only for binary classification.
    proba = 1.0 / (1.0 + np.exp(-logits_test))

    # radar_audio: remove thresholding if regression.
    pred = (proba >= 0.5).astype(int)

    # radar_audio: replace with MAE/RMSE/R² if regression.
    accuracy = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, zero_division=0)

    if len(np.unique(y_test)) > 1:
        roc_auc = roc_auc_score(y_test, proba)
    else:
        roc_auc = np.nan
        print("Warning: ROC-AUC is NaN because the test set has only one class.")

    results = {
        "backbone": DL_BACKBONE,
        "n_rows": len(y_arr),
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "n_groups": len(np.unique(group_arr)),
        "n_train_groups": len(np.unique(group_arr[train_idx])),
        "n_test_groups": len(np.unique(group_arr[test_idx])),
        "epochs": DL_EPOCHS,
        "batch_size": DL_BATCH,
        "learning_rate": DL_LR,

        # radar_audio: replace these metrics if regression.
        "accuracy": accuracy,
        "f1": f1,
        "roc_auc": roc_auc,
    }

    results_df = pd.DataFrame([results])

    # radar_audio: rename output tag.
    # Example:
    # tag = f"radar_audio_{DL_BACKBONE}_dl_head_no_folds"
    tag = f"androids_{DL_BACKBONE}_dl_head_no_folds"
    summary_path = RESULTS_PATH / f"{tag}_summary.csv"
    predictions_path = RESULTS_PATH / f"{tag}_predictions.csv"

    results_df.to_csv(summary_path, index=False)

    pred_df = pd.DataFrame(
        {
            "file_path": path_arr[test_idx],

            # radar_audio: should still be participant_id.
            "group": group_arr[test_idx],

            # radar_audio: if regression, y_true becomes PHQ-8 score.
            "y_true": y_test.astype(int),

            # radar_audio: rename to "prediction" or "predicted_score"
            # if regression.
            "probability": proba,

            # radar_audio: remove for regression.
            "prediction": pred.astype(int),
        }
    )

    pred_df.to_csv(predictions_path, index=False)

    print()
    print(results_df)
    print()
    print("Saved summary:", summary_path)
    print("Saved predictions:", predictions_path)

    model.cpu()
    del model, optimizer, loss_fn, train_loader, test_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

run_dl_androids_no_folds()